In [1]:
import os 
import warnings
warnings.filterwarnings("ignore")
import autogen
from autogen import ConversableAgent
from autogen.agentchat.contrib.multimodal_conversable_agent import MultimodalConversableAgent
from IPython.display import Markdown, display, Image
from getpass import getpass


In [2]:
OPENAI_KEY = getpass('Enter Open AI API Key: ')
os.environ["OPENAI_API_KEY"] = OPENAI_KEY

In [15]:
llm_config = {
    "config_list": [{"model": "gpt-4.1", "api_key": os.environ["OPENAI_API_KEY"],
                     "temperature": 0.7, "cache_seed": None}],
}

In [16]:
bill_processing_agent = MultimodalConversableAgent(
    name="Bill_Processing_Agent",
    system_message="""
You are a Bill Processing Agent.

Role:
- Analyze uploaded images of bills and receipts.
- Extract structured financial information from each image.

Your responsibilities:
1. Identify and extract the following details from the bill image:
   - Merchant / Store name
   - Date of transaction (if visible)
   - Individual line items (if readable)
   - Total amount paid
   - Tax amount (if available)
   - Payment method (cash/card/UPI if visible)

2. Categorize each expense into one of the following categories:
   - Groceries
   - Dining
   - Utilities
   - Shopping
   - Entertainment
   - Transportation
   - Healthcare
   - Others (if none apply)

3. If multiple bills are provided:
   - Process each bill separately
   - Aggregate totals by category
   - Calculate the overall total spend

Output format:
- A structured, easy-to-read summary including:
   - List of expenses grouped by category
   - Total amount per category
   - Grand total amount spent

Guidelines:
- If any field is not visible, clearly mark it as "Not available".
- Do NOT make assumptions about unclear or unreadable values.
- Be precise with numbers and currency symbols as shown in the image.
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)


In [17]:
expense_summarization_agent = ConversableAgent(
    name="Expense_Summarization_Agent",
    system_message="""
You are an Expense Summarization Agent.

Role:
- Analyze categorized expense data received from the Bill Processing Agent.
- Generate spending insights and trends.

Expected Input:
- A categorized list of expenses with:
  - Category name
  - Individual expenses
  - Total amount per category
  - Grand total spend

Your responsibilities:
1. Calculate total spending per category.
2. Identify:
   - The highest spending category
   - Categories with unusually high spend relative to others
3. Highlight noticeable patterns, such as:
   - Heavy spending concentration in a single category
   - Multiple categories with similar high spend
   - One-time large expenses vs frequent small expenses

Output format:
- Clear, human-readable summary including:
  - Spending breakdown by category
  - Highest expenditure category
  - Any unusual or noteworthy spending trends

Guidelines:
- Do NOT modify or re-categorize expenses.
- Do NOT invent new data or assumptions.
- If trends cannot be confidently identified, explicitly state so.
- Keep the summary concise, factual, and insight-driven.
""",
    llm_config=llm_config,
    human_input_mode="NEVER",
)


In [18]:
user_proxy = autogen.UserProxyAgent(
    name="User_Proxy",
    system_message="""
You represent the end user in an expense analysis system.

Role:
- Provide images of bills and receipts to the agent group.
- Coordinate with agents to extract, categorize, and summarize expenses.

Responsibilities:
- Forward uploaded bill or receipt images to the Group Manager or relevant agents.
- Clarify user intent if required (e.g., time range, spending focus).
- Terminate the conversation once a complete expense summary and insights are produced.

Guidelines:
- Do NOT analyze images yourself.
- Do NOT modify extracted data.
- Allow agents to autonomously process, summarize, and analyze expenses.
- If URLs or image references are provided, pass them as-is for agent processing.
""",
    human_input_mode="TERMINATE",  # User intervenes only if clarification is required
    max_consecutive_auto_reply=0,
    code_execution_config={
        "use_docker": False
    },
)


In [19]:
group_meet = autogen.GroupChat(
    agents=[user_proxy,bill_processing_agent, expense_summarization_agent],
    messages=[],
    max_round=4,
    speaker_selection_method= "auto",
    select_speaker_auto_llm_config = llm_config
)
group_manager = autogen.GroupChatManager(
    groupchat=group_meet,
)

In [26]:
import base64

def image_to_base64(image_path):
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

image_b64 = image_to_base64("bill_.jpeg")

In [34]:
bill = user_proxy.initiate_chat(
    group_manager,
    message="""
Analyze the uploaded bill/receipt image and extract expense details.
Categorize the expenses and provide a summarized spending report.

<img https://i.ytimg.com/vi/CarUZqvtzeU/maxresdefault.jpg>
""",
    summary_method="last_msg",
)


User_Proxy (to chat_manager):


Analyze the uploaded bill/receipt image and extract expense details.
Categorize the expenses and provide a summarized spending report.

<img https://i.ytimg.com/vi/CarUZqvtzeU/maxresdefault.jpg>


--------------------------------------------------------------------------------



Next speaker: Bill_Processing_Agent

Bill_Processing_Agent (to chat_manager):

Bill Analysis & Expense Extraction

1. Extracted Bill Details

- Merchant / Store name: McDonald's
- Date of transaction: Not available (not visible in the image)
- Individual line items (as visible and readable):
  - McAloo Tikki Burger: ₹32.00
  - Medium Fries: ₹74.00
  - Medium Coke: ₹62.00
- Total amount paid: ₹168.00
- Tax amount: Not available (not visible in the image)
- Payment method: Not available (not visible in the image)

2. Categorization of Expenses

All listed items are food and beverage purchases from a restaurant.

- Category: Dining

3. Structured Expense Summary

| Category | Items                                                               | Total Amount |
|----------|---------------------------------------------------------------------|--------------|
| Dining   | McAloo Tikki Burger, Medium Fries, Medium Coke                      | ₹168.00      |

Grand Total Amount Spent: ₹168.00

